 <h1 style="color:black; background-color:lightgreen; text-align:center;">Թարգմանություն Google Cloud Translation API-ի միջոցով / Translation model via Google Cloud Translation API</h1>

### HY: Սա մոդել Է տեքստերի ավտոմատ թարգմանության համար՝ Google Cloud Translation API-ն օգտագործելով։ Այն աջակցում է բազմալեզու թարգմանություններին և ապահովում է մշակված պատասխանի ձևեր օգտատիրոջ հարցումների համար(ֆայլը՝ translate_response.py)։
### Google Cloud Translation API-ի համար օգտագործվում է API key, որը ունենալու համար պետք է գրանցվել Google Cloud Platform-ում, ստեղծել պրոյեկտ ու ինտեգրել  Cloud Translation API-ը։ 
### **Օգտագործման ձև**
**Տեքստ, հիմնական լեզու և թիրախային լեզու տրամադրել translate ֆունկցիային։<br>
Կոդը կստուգի լեզուների վավերականությունը։<br>
Կկատարվի API կանչ՝ թարգմանության համար։<br>
Կվերադարձվի թարգմանված տեքստը կամ համապատասխան սխալի հաղորդագրությունը։**

### EN: This is a model for automatic text translation using the Google Cloud Translation API. It supports multilingual translations and provides  response formats for user requests (file: translate_response.py).
### The Google Cloud Translation API uses an API key, which you need to register with Google Cloud Platform, create a project, and integrate the Cloud Translation API.
### **Usage**
**Provide text, a base language, and a target language to the translate function.<br>
The code will check the validity of the languages.<br>
An API call will be made for the translation.<br>
The translated text or an appropriate error message will be returned.**

In [1]:
import sys
import os

# Add the current directory to the Python path
sys.path.append(os.getcwd())

# Now you can safely import your modules
import language_list
import translate_response

import json
import random
from http.client import HTTPException
from urllib.error import HTTPError, URLError

from googleapiclient.discovery import build

from language_list import _LANGUAGE_CODE_LIST as language_code_dict
from language_list import _LANGUAGE_LIST as language_dict
from translate_response import (_TRANSLATE_ERROR, _TRANSLATE_INTO_W,
                                _TRANSLATE_NETWORK_ERROR, _TRANSLATE_RESULT,
                                _TRANSLATE_UNKNOWN_LANGUAGE, _TRANSLATE_W,
                                _TRANSLATE_W_FROM, _TRANSLATE_W_FROM_TO,
                                _TRANSLATE_W_TO)


API_KEY = 'your API key'
TRANSLATION_SERVICE = build('translate', 'v2', developerKey=API_KEY)

def translate(text, source_lang, target_lang):
    """Returns a string containing translated text, or a request for more info

    Takes text input, source and target language for the text (all strings)
    uses the responses found in translate_response.py as templates
    """

    # Validate the languages provided by the user
    source_lang_code = validate_language(source_lang)
    target_lang_code = validate_language(target_lang)

    # If both languages are invalid or no languages are provided tell the user
    if not source_lang_code and not target_lang_code:
        response = random.choice(_TRANSLATE_UNKNOWN_LANGUAGE)

    # If there is no text but two valid languages ask the user for input
    if not text and source_lang_code and target_lang_code:
        response = random.choice(_TRANSLATE_W_FROM_TO).format(
            lang_from=language_code_dict[source_lang_code],
            lang_to=language_code_dict[target_lang_code])

    # If there is no text but a valid target language ask the user for input
    if not text and target_lang_code:
        response = random.choice(_TRANSLATE_W_TO).format(
            lang=language_code_dict[target_lang_code])

    # If there is no text but a valid source language assume the target
    # language is English if the source language is not English
    if (not text and
        source_lang_code and
        source_lang_code != 'en' and
            not target_lang_code):
        target_lang_code = 'en'

    # If there is no text, no target language and the source language is English
    # ask the user for text
    if (not text and
        source_lang_code and
        source_lang_code == 'en' and
            not target_lang_code):
        response = random.choice(_TRANSLATE_W_FROM).format(
            lang=language_code_dict[source_lang_code])

    # If there is no text and no languages
    if not text and not source_lang_code and not target_lang_code:
        response = random.choice(_TRANSLATE_W)

    # If there is text but no languages
    if text and not source_lang_code and not target_lang_code:
        response = random.choice(_TRANSLATE_INTO_W)

    # If there is text and a valid target language but no source language
    if text and not source_lang_code and target_lang_code:
        response = translate_text(text, source_lang_code, target_lang_code)

    # If there is text and 2 valid languages return the translation
    if text and source_lang_code and target_lang_code:
        response = translate_text(text, source_lang_code, target_lang_code)

    # If no response is generated from the any of the 8 possible combinations
    # (3 booleans = 2^3 = 8 options) return an error to the user
    if not response:
        response = random.choice(_TRANSLATE_ERROR)

    return response


def translate_text(query, source_lang_code, target_lang_code):
    """returns translated text or text indicating a translation/network error

    Takes a text to be translated, source language and target language code
    2 letter ISO code found in language_list.py
    """

    try:
        translations = TRANSLATION_SERVICE.translations().list(
            source=source_lang_code,
            target=target_lang_code,
            q=query
        ).execute()
        translation = translations['translations'][0]
        if 'detectedSourceLanguage' in translation.keys():
            source_lang_code = translation['detectedSourceLanguage']
        resp = random.choice(_TRANSLATE_RESULT).format(
            text=translation['translatedText'],
            fromLang=language_code_dict[source_lang_code],
            toLang=language_code_dict[target_lang_code])
    except (HTTPError, URLError, HTTPException):
        resp = random.choice(_TRANSLATE_NETWORK_ERROR)
    except Exception:
        resp = random.choice(_TRANSLATE_ERROR)
    return resp


def validate_language(language):
    """returns 2 letter language code if valid, None if language is invalid

    Uses dictionary in language_list.py to verify language is valid
    """

    try:
        lang_code = language_dict[language]
    except KeyError:
        lang_code = None
    return lang_code

if __name__ == '__main__':
    # Example input
    text_to_translate = "Hello, how are you?"
    source_language = "English"
    target_language = "Armenian"

    # Perform translation
    translation_result = translate(text_to_translate, source_language, target_language)
    print(f"Translation Result: {translation_result}")

    # Example input for reverse translation
    text_in_armenian = "Բարև, ինչպե՞ս ես:"
    source_language = "Armenian"
    target_language = "English"

    # Perform reverse translation
    reverse_translation_result = translate(text_in_armenian, source_language, target_language)
    print(f"Reverse Translation Result: {reverse_translation_result}")

Translation Result: Here is how that translates from English into Armenian: Բարև ձեզ ինչպես եք.
Reverse Translation Result: Here is how that translates from Armenian into English: Hello, how are you?.


In [3]:
if __name__ == '__main__':
    # Example input
    text_to_translate = "Artificial intelligence is the future of technology."
    source_language = "English"
    target_language = "Armenian"

    # Perform translation
    translation_result = translate(text_to_translate, source_language, target_language)
    print(f"Translation Result: {translation_result}")

    # Example input for reverse translation
    text_in_armenian =  "Տեխնոլոգիայի ապագան գտնվում է արհեստական բանականության մեջ:"
    source_language = "Armenian"
    target_language = "English"

    # Perform reverse translation
    reverse_translation_result = translate(text_in_armenian, source_language, target_language)
    print(f"Reverse Translation Result: {reverse_translation_result}")

Translation Result: Here is how that translates from English into Armenian: Արհեստական ինտելեկտը տեխնոլոգիայի ապագան է։.
Reverse Translation Result: That translates from Armenian to English like so: The future of technology lies in artificial intelligence..
